In [ ]:
import streamlit as st
import ollama
import PIL import Image
import io
import base64

In [ ]:
# Page config
st.set_page_config(page_title="Image Text Extractor", layout="wide")

# ---------- HEADER ----------
st.markdown("""
    <div style="display:flex; align-items:center; gap:10px;">
        <h1 style="margin-bottom:0;">🧾 Image Text Extractor</h1>
    </div>
    <p style="color:gray; margin-top:0;">
        Upload an image and convert visible text into structured Markdown.
    </p>
""", unsafe_allow_html=True)

st.divider()

# ---------- SIDEBAR ----------
with st.sidebar:
    st.markdown("## ⚙️ Controls")

    uploaded_file = st.file_uploader(
        "Select an image file",
        type=["png", "jpg", "jpeg"]
    )

    if uploaded_file:
        image = Image.open(uploaded_file)
        st.image(image, caption="Preview", use_column_width=True)

        extract = st.button("Run OCR", use_container_width=True)

    else:
        extract = False

    st.markdown("---")

    if st.button("Reset Output"):
        st.session_state.pop("ocr_result", None)
        st.rerun()

# MAIN CONTENT
if extract:
    with st.spinner("Reading image"):
        try:
            response = ollama.chat(
                model='gemma3:12b',
                messages=[{
                    'role': 'user',
                    'content': """Extract all readable text from this image.
                                  Format the result using clean Markdown with
                                  proper headings, lists, or code blocks where appropriate.""",
                    'images': [uploaded_file.getvalue()]
                }]
            )
            st.session_state["ocr_result"] = response.message.content
            
        except Exception as e:
            st.error(f"Error during OCR: {e}")
            st.session_state.pop("ocr_result", None)
